# Final Model Evaluation on Test Set (QUICK TEST VERSION)

**NOTE: This is a quick test version that only uses a subset of test files for fast execution.**

Evaluate the best models from each approach:
1. AutoGluon (Hybrid Model)
2. Naive Baseline

In [1]:
import os
import sys
import json
import glob
import time
import pandas as pd
import numpy as np
import torch
import random
from tqdm.auto import tqdm
from chronos import Chronos2Pipeline
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')

if os.path.basename(os.getcwd()) == 'notebooks':
    project_root = os.path.abspath('..')
else:
    project_root = os.getcwd()

if project_root not in sys.path:
    sys.path.append(project_root)

from src.datamodule import ElectricityDataModule

## 1. Configuration

In [2]:
BASE_DIR = ".."
DATA_DIR = os.path.join(BASE_DIR, "data")
TEST_DIR = os.path.join(DATA_DIR, "test")
TEST_DIR_QUICK = os.path.join(DATA_DIR, "test_quick")
SCALERS_DIR = os.path.join(DATA_DIR, "scalers")
MODELS_DIR = os.path.join(BASE_DIR, "models")
RESULTS_DIR = os.path.join(BASE_DIR, "results")

os.makedirs(RESULTS_DIR, exist_ok=True)

TARGET_COLS = ["high", "low", "close", "volume"]
INPUT_CHUNK_LENGTH = 48
OUTPUT_CHUNK_LENGTH = 10
SEED = 827
MAX_TEST_FILES = 10 # Set None to load all files
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define covariates - these will be returned by the dataloader
PAST_COVARIATES = [
    'close_lag_adj_1', 'close_delta_adj_1', 'volume_adj_1',
    'close_lag_adj_2', 'close_delta_adj_2', 'volume_adj_2',
    'close_lag_adj_3', 'close_delta_adj_3', 'volume_adj_3',
    'close_lag_adj_4', 'close_delta_adj_4', 'volume_adj_4',
    'close_lag_adj_5', 'close_delta_adj_5', 'volume_adj_5',
    'close_lag_adj_6', 'close_delta_adj_6', 'volume_adj_6',
    'nearest_liquid_contract_close', 'cross_contract_mean',
    'is_trading'
]
FUTURE_COVARIATES = [
    'time_to_delivery', 'hour_of_day', 'day_of_week',
    'week_of_year', 'month', 'is_weekend',
]

# Regressor covariates (exclude is_trading since it's classifier's target)
REGRESSOR_PAST_COVARIATES = [c for c in PAST_COVARIATES if c != 'is_trading']

print('='*80)
print(f"QUICK TEST MODE: Using {MAX_TEST_FILES} test files")
print('='*80)

test_files = sorted(glob.glob(os.path.join(TEST_DIR, "*.parquet")))
random.seed(SEED)
random.shuffle(test_files)

if MAX_TEST_FILES is not None:
    test_files = test_files[:MAX_TEST_FILES]
    print('='*80)
    print(f"QUICK TEST MODE: Using {MAX_TEST_FILES} test files")
    print('='*80)

print('='*80)
print("Configuration loaded successfully")
print(f"Test data directory: {TEST_DIR}")
print(f"Number of test files: {len(test_files)}")
print(f"Device: {device}")
print(f"Past covariates: {len(PAST_COVARIATES)}")
print(f"Future covariates: {len(FUTURE_COVARIATES)}")
print('='*80)

QUICK TEST MODE: Using 10 test files
QUICK TEST MODE: Using 10 test files
Configuration loaded successfully
Test data directory: ..\data\test
Number of test files: 10
Device: cuda
Past covariates: 21
Future covariates: 6


In [3]:
def simple_smape(y_true, y_pred, eps=1e-6):
    num = torch.abs(y_true - y_pred)
    den = torch.abs(y_true) + torch.abs(y_pred) + eps
    smape = 2.0 * num / den
    return smape.mean()

## 2. Evaluate Naive Baseline

In [4]:
print('='*80)
print("EVALUATING NAIVE BASELINE")
print('='*80)

start = time.time()

# Create datamodule WITHOUT covariates for naive baseline (doesn't need them)
datamodule = ElectricityDataModule(
    train_parquet=os.path.join(DATA_DIR, "train"),
    val_parquet=os.path.join(DATA_DIR, "val"),
    test_parquet=TEST_DIR,
    scalers_dir=SCALERS_DIR,
    batch_size=32,
    num_workers=4,
    dataset_kwargs={
        'stride': OUTPUT_CHUNK_LENGTH  # Non-overlapping windows
    }
)
datamodule.setup(stage='test')
test_dataloader = datamodule.test_dataloader()

all_naive_forecasts = []
all_ground_truths = []

for batch in tqdm(test_dataloader, desc="Naive forecasting"):
    past, _, future, _, _, _ = batch
    last_values = past[:, -1:, :]
    naive_pred = last_values.repeat(1, OUTPUT_CHUNK_LENGTH, 1)
    all_naive_forecasts.append(naive_pred)
    all_ground_truths.append(future)

naive_forecasts = torch.cat(all_naive_forecasts, dim=0).to(device)
naive_ground_truths = torch.cat(all_ground_truths, dim=0).to(device)

naive_smape = simple_smape(naive_ground_truths, naive_forecasts).item() * 100
naive_by_target = {}
for i, target_name in enumerate(TARGET_COLS):
    target_smape = simple_smape(naive_ground_truths[:, :, i], naive_forecasts[:, :, i]).item() * 100
    naive_by_target[target_name] = target_smape

print(f"Evaluated based on {MAX_TEST_FILES if MAX_TEST_FILES is not None else 'all'} files in {((time.time() - start)/60):.4f} minutes")
print(f"Naive Baseline sMAPE: {naive_smape:.2f}%")
for target_name, smape in naive_by_target.items():
    print(f"  {target_name:8s}: {smape:.2f}%")
print('='*80)

EVALUATING NAIVE BASELINE


Naive forecasting: 0it [00:00, ?it/s]

Evaluated based on 10 files in 3.1133 minutes
Naive Baseline sMAPE: 7.06%
  high    : 4.61%
  low     : 4.65%
  close   : 4.65%
  volume  : 14.32%


## 3. Evaluate AutoGluon Hybrid Model

In [5]:
# Create a second datamodule WITH covariates for AutoGluon
print("Creating dataloader with covariates for Chronos2 and AutoGluon...")
datamodule_with_covariates = ElectricityDataModule(
    train_parquet=os.path.join(DATA_DIR, "train"),
    val_parquet=os.path.join(DATA_DIR, "val"),
    test_parquet=TEST_DIR,
    scalers_dir=SCALERS_DIR,
    batch_size=32,
    num_workers=4,
    dataset_kwargs={
        'return_covariates': True,
        'past_covariate_cols': PAST_COVARIATES,
        'future_covariate_cols': FUTURE_COVARIATES,
        'stride': OUTPUT_CHUNK_LENGTH  # Non-overlapping windows
    }
)
datamodule_with_covariates.setup(stage='test')
test_dataloader_with_covariates = datamodule_with_covariates.test_dataloader()
print("✓ Created dataloader with covariates")
print(f"  Input chunk length: {INPUT_CHUNK_LENGTH}")
print(f"  Output chunk length: {OUTPUT_CHUNK_LENGTH}")
print(f"  Stride: {OUTPUT_CHUNK_LENGTH} (non-overlapping)")

Creating dataloader with covariates for Chronos2 and AutoGluon...
✓ Created dataloader with covariates
  Input chunk length: 48
  Output chunk length: 10
  Stride: 10 (non-overlapping)


In [ ]:
print("="*80)
print("EVALUATING AUTOGLUON HYBRID MODEL (OPTIMIZED)")
print("="*80)

start = time.time()

AUTOGLUON_CLASSIFIER_DIR = os.path.join(MODELS_DIR, "autogluon_trading_classifier")
AUTOGLUON_REGRESSOR_DIR = os.path.join(MODELS_DIR, "autogluon_hybrid_regressor")

autogluon_hybrid_smape = float('inf')
autogluon_hybrid_by_target = {}

# Day mapping for time_to_delivery calculation (same as preprocess.py)
DAY_MAP = {"Mon": 0, "Tue": 1, "Wed": 2, "Thu": 3, "Fri": 4, "Sat": 5, "Sun": 6}

def get_delivery_details(asset_name):
   """Extract delivery details from asset name (same logic as preprocess.py)"""
   day_str = asset_name[:3]
   hour = int(asset_name[3:5])
   quarter = int(asset_name[6:])
   minute = (quarter - 1) * 15
   delivery_day_of_week = DAY_MAP[day_str]
   return delivery_day_of_week, hour, minute

def calculate_time_to_delivery(timestamp, asset_name):
   """Calculate time_to_delivery (same logic as preprocess.py)"""
   delivery_day, hour, minute = get_delivery_details(asset_name)

   # Calculate days ahead to next delivery
   days_ahead = (delivery_day - timestamp.dayofweek + 7) % 7
   delivery_date = timestamp.normalize() + pd.Timedelta(days=days_ahead)
   delivery_datetime = delivery_date + pd.Timedelta(hours=hour, minutes=minute)

   # If delivery time has passed this week, target next week
   if delivery_datetime <= timestamp:
       delivery_datetime += pd.Timedelta(days=7)

   # Calculate difference in hours
   time_to_delivery = (delivery_datetime - timestamp).total_seconds() / 3600.0
   return max(0, time_to_delivery)

if os.path.exists(AUTOGLUON_CLASSIFIER_DIR) and os.path.exists(AUTOGLUON_REGRESSOR_DIR):
   try:
       print(f"Loading AutoGluon hybrid models...")
       classifier_predictor = TimeSeriesPredictor.load(AUTOGLUON_CLASSIFIER_DIR)
       regressor_predictor = TimeSeriesPredictor.load(AUTOGLUON_REGRESSOR_DIR)
       print(f"✓ Loaded classifier and regressor predictors.")

       all_hybrid_forecasts = []
       all_hybrid_gts = []

       # Find is_trading index in past covariates
       is_trading_idx = PAST_COVARIATES.index('is_trading')

       for batch in tqdm(test_dataloader_with_covariates, desc="AutoGluon Hybrid Forecasting"):
           # Unpack 8 elements (with covariates)
           past, past_mask, future, future_mask, asset_ids, past_ts_list, past_covariates, future_covariates = batch
           batch_size = past.shape[0]

           clf_rows, reg_rows, known_cov_rows = [], [], []

           for i in range(batch_size):
               item_id_base = f"window_{i}"

               # Create timestamps with 15-minute frequency (to match training data frequency)
               base_time = pd.Timestamp('2024-01-01 00:00:00')
               timestamps = pd.date_range(start=base_time, periods=INPUT_CHUNK_LENGTH, freq='15min')
               future_timestamps = pd.date_range(start=timestamps[-1] + pd.Timedelta(minutes=15), periods=OUTPUT_CHUNK_LENGTH, freq='15min')

               # Extract is_trading from past covariates
               is_trading_series = past_covariates[i, :, is_trading_idx].cpu().numpy()

               # Get asset_id for time_to_delivery calculation
               asset_id = asset_ids[i]

               # Build classifier INPUT data (ALL 48 timesteps)
               for t in range(INPUT_CHUNK_LENGTH):
                   row = {'item_id': item_id_base, 'timestamp': timestamps[t], 'is_trading': is_trading_series[t]}

                   # Add target columns (high, low, close, volume)
                   for j, col_name in enumerate(TARGET_COLS):
                       row[col_name] = past[i, t, j].item()

                   # Add all past covariates EXCEPT is_trading
                   for j, cov_name in enumerate(PAST_COVARIATES):
                       if cov_name != 'is_trading':
                           row[cov_name] = past_covariates[i, t, j].item()

                   # Add time-based future covariates (compute from timestamp)
                   row['hour_of_day'] = timestamps[t].hour
                   row['day_of_week'] = timestamps[t].dayofweek
                   row['week_of_year'] = timestamps[t].isocalendar()[1]
                   row['month'] = timestamps[t].month
                   row['is_weekend'] = 1.0 if timestamps[t].dayofweek >= 5 else 0.0
                   row['time_to_delivery'] = calculate_time_to_delivery(timestamps[t], asset_id)
                   row['block'] = 0.0

                   clf_rows.append(row)

               # Build regressor INPUT data (ALL 48 timesteps - same as classifier)
               # Even though regressor was trained on trading-only, it receives all data during inference
               for j, col_name in enumerate(TARGET_COLS):
                   reg_item_id = f"{item_id_base}_{col_name}"

                   for t in range(INPUT_CHUNK_LENGTH):
                       row = {'item_id': reg_item_id, 'timestamp': timestamps[t], 'target': past[i, t, j].item()}

                       # Add past covariates EXCEPT is_trading
                       for k, cov_name in enumerate(REGRESSOR_PAST_COVARIATES):
                           past_cov_idx = PAST_COVARIATES.index(cov_name)
                           row[cov_name] = past_covariates[i, t, past_cov_idx].item()

                       # Add time-based features
                       row['hour_of_day'] = timestamps[t].hour
                       row['day_of_week'] = timestamps[t].dayofweek
                       row['week_of_year'] = timestamps[t].isocalendar()[1]
                       row['month'] = timestamps[t].month
                       row['is_weekend'] = 1.0 if timestamps[t].dayofweek >= 5 else 0.0
                       row['time_to_delivery'] = calculate_time_to_delivery(timestamps[t], asset_id)
                       row['block'] = 0.0

                       reg_rows.append(row)

               # Build known covariates for FUTURE window (same for both classifier and regressor)
               for t_idx, t in enumerate(future_timestamps):
                   # Classifier known covariates
                   row = {'item_id': item_id_base, 'timestamp': t}
                   for k, cov_name in enumerate(FUTURE_COVARIATES):
                       row[cov_name] = future_covariates[i, t_idx, k].item()
                   known_cov_rows.append(row)

                   # Regressor known covariates (one per target column)
                   for j, col_name in enumerate(TARGET_COLS):
                       reg_item_id = f"{item_id_base}_{col_name}"
                       row = {'item_id': reg_item_id, 'timestamp': t}
                       for k, cov_name in enumerate(FUTURE_COVARIATES):
                           row[cov_name] = future_covariates[i, t_idx, k].item()
                       known_cov_rows.append(row)

           # Predict is_trading for future window
           classifier_ts = TimeSeriesDataFrame.from_data_frame(pd.DataFrame(clf_rows), id_column='item_id', timestamp_column='timestamp')

           # Predict price/volume for future window
           regressor_ts = TimeSeriesDataFrame.from_data_frame(pd.DataFrame(reg_rows), id_column='item_id', timestamp_column='timestamp')

           # Known covariates (shared)
           known_cov_df = pd.DataFrame(known_cov_rows)
           known_cov_ts = TimeSeriesDataFrame.from_data_frame(known_cov_df, id_column='item_id', timestamp_column='timestamp')

           # Get predictions
           is_trading_forecasts = classifier_predictor.predict(classifier_ts, model="DeepAR", known_covariates=known_cov_ts)
           price_volume_forecasts = regressor_predictor.predict(regressor_ts, model="TemporalFusionTransformer_FineTuned", known_covariates=known_cov_ts)

           # Combine predictions: final = classifier × regressor
           batch_final_preds = torch.zeros_like(future)

           for i in range(batch_size):
               item_id_base = f"window_{i}"

               # Get classifier predictions (binary: 0 or 1 for is_trading)
               trading_preds = is_trading_forecasts.loc[item_id_base]['mean'].values
               trading_mask = (trading_preds >= 0.5).astype(float)  # Convert to 0/1

               # Get regressor predictions for each target
               for j, col_name in enumerate(TARGET_COLS):
                   regressor_item_id = f"{item_id_base}_{col_name}"
                   if regressor_item_id in price_volume_forecasts.item_ids:
                       reg_values = price_volume_forecasts.loc[regressor_item_id]['mean'].values
                       # Hybrid prediction: multiply by trading mask
                       batch_final_preds[i, :, j] = torch.from_numpy(reg_values * trading_mask).float()

           all_hybrid_forecasts.append(batch_final_preds)
           all_hybrid_gts.append(future)

       hybrid_forecasts = torch.cat(all_hybrid_forecasts, dim=0).to(device)
       hybrid_gts = torch.cat(all_hybrid_gts, dim=0).to(device)

       autogluon_hybrid_smape = simple_smape(hybrid_gts, hybrid_forecasts).item() * 100
       autogluon_hybrid_by_target = {}
       for i, target_name in enumerate(TARGET_COLS):
           autogluon_hybrid_by_target[target_name] = simple_smape(hybrid_gts[:, :, i], hybrid_forecasts[:, :, i]).item() * 100

       print(f"Evaluated based on {MAX_TEST_FILES if MAX_TEST_FILES is not None else 'all'} files in {((time.time() - start)/60):.4f} minutes")
       print(f"\nAutoGluon Hybrid Model sMAPE: {autogluon_hybrid_smape:.2f}%")
       for target_name, smape in autogluon_hybrid_by_target.items():
           print(f"  {target_name:8s}: {smape:.2f}%")

   except Exception as e:
       print(f"✗ Error evaluating AutoGluon hybrid model: {e}")
       import traceback
       traceback.print_exc()
else:
   print(f"✗ AutoGluon hybrid models not found. Run the training notebook first.")

print("="*80)

Loading predictor from path C:\Users\merta\OneDrive\Desktop\FS Courses\Deep Learning\Final_Project\models\autogluon_trading_classifier
Loading predictor from path C:\Users\merta\OneDrive\Desktop\FS Courses\Deep Learning\Final_Project\models\autogluon_hybrid_regressor


EVALUATING AUTOGLUON HYBRID MODEL (OPTIMIZED)
Loading AutoGluon hybrid models...
✓ Loaded classifier and regressor predictors.


AutoGluon Hybrid Forecasting: 0it [00:00, ?it/s]

[INFO] Forecast is not sample based. Ignoring parameter `num_samples` from predict method.


## 4. Final Comparison and Results

In [ ]:
logging.info("--- Final Results ---")
final_results = {
    'AutoGluon Hybrid': autogluon_hybrid_smape,
    'Naive Baseline': naive_smape,
}
sorted_results = sorted(final_results.items(), key=lambda x: x[1])

print("Overall sMAPE (ranked):")
for rank, (model, smape) in enumerate(sorted_results, 1):
    if smape != float('inf'):
        print(f"  {rank}. {model:25s}: {smape:.2f}%")
    else:
        print(f"  {rank}. {model:25s}: N/A")

best_model_name, best_smape = sorted_results[0]
print(f"\nBEST MODEL: {best_model_name} with sMAPE: {best_smape:.2f}%")